In [ ]:
import datetime

import holidays
import numpy as np
import pandas as pd

In [ ]:
def get_norwegian_public_holidays(year):
    """
    henter norske offentlige fridager for et gitt år.
    """
    norwegian_holidays = holidays.norway(years=[year], language="no")
    important_days = {
        "første nyttårsdag",
        "skjærtorsdag",
        "langfredag",
        "første påskedag",
        "andre påskedag",
        "arbeidernes dag",
        "grunnlovsdag",
        "kristi himmelfartsdag",
        "første pinsedag",
        "andre pinsedag",
        "første juledag",
        "andre juledag",
    }
    days_off = [
        date.strftime("%Y-%m-%d")
        for date, name in norwegian_holidays.items()
        if date.year == year and name in important_days
    ]
    return np.array(days_off)

In [ ]:
def is_norwegian_holiday(date, holidays_array):
    """
    Sjekker om en dato er en norsk fridag. Returnerer 1 eller 0.
    """
    date = pd.to_datetime(date) if not isinstance(date, pd.Timestamp) else date
    return 1 if date.strftime("%Y-%m-%d") in holidays_array else 0

In [ ]:
def get_european_public_holidays(country_code, year):
    """
    Get major public holidays for European countries.
    Similar to Norwegian holidays but adapted for each country.
    """
    # Map country names to proper holiday library codes
    country_mapping = {
        'sweden': 'SE',
        'denmark': 'DK', 
        'germany': 'DE',
        'france': 'FR',
        'uk': 'GB',
        'netherlands': 'NL',
        'spain': 'ES',
        'italy': 'IT'
    }
    
    # Key holidays mapping using actual local language names from the library
    holiday_mappings = {
        'sweden': {
            'nyårsdagen', 'trettondedag jul', 'långfredagen', 'påskdagen; söndag', 'annandag påsk',
            'första maj', 'kristi himmelsfärdsdag', 'sveriges nationaldag', 'pingstdagen; söndag',
            'midsommarafton', 'juldagen', 'annandag jul'
        },
        'denmark': {
            'nytårsdag', 'skærtorsdag', 'langfredag', 'påskedag', 'anden påskedag',
            'store bededag', 'kristi himmelfartsdag', 'pinsedag', 'anden pinsedag',
            'juledag', 'anden juledag'
        },
        'germany': {
            'neujahr', 'karfreitag', 'ostermontag', 'erster mai', 'christi himmelfahrt',
            'pfingstmontag', 'tag der deutschen einheit', 'erster weihnachtstag', 'zweiter weihnachtstag'
        },
        'france': {
            'jour de l\'an', 'lundi de pâques', 'fête du travail', 'fête de la victoire',
            'ascension', 'lundi de pentecôte', 'fête nationale', 'assomption',
            'toussaint', 'armistice'
        },
        'uk': {
            'new year\'s day', 'good friday', 'easter monday', 'early may bank holiday',
            'spring bank holiday', 'summer bank holiday', 'christmas day', 'boxing day'
        },
        'netherlands': {
            'nieuwjaarsdag', 'goede vrijdag', 'paaszondag', 'paasmaandag',
            'koningsdag', 'bevrijdingsdag', 'hemelvaartsdag', 'pinksterzondag', 'pinkstermaandag',
            'eerste kerstdag', 'tweede kerstdag'
        },
        'spain': {
            'año nuevo', 'epifanía del señor', 'viernes santo', 'fiesta del trabajo',
            'asunción de la virgen', 'fiesta nacional de españa', 'todos los santos',
            'día de la constitución española', 'inmaculada concepción', 'navidad'
        },
        'italy': {
            'capodanno', 'epifania', 'lunedì dell\'angelo', 'festa della liberazione',
            'festa del lavoro', 'festa della repubblica', 'assunzione di maria vergine',
            'ognissanti', 'immacolata concezione', 'natale di gesù', 'santo stefano'
        }
    }
    
    # Get the proper country code
    country_lower = country_code.lower()
    if country_lower in country_mapping:
        proper_code = country_mapping[country_lower]
    else:
        proper_code = country_code.upper()
    
    try:
        # Get holidays in local language (default)
        country_holidays = holidays.country_holidays(proper_code, years=[year])
    except NotImplementedError:
        print(f"Warning: Country {country_code} not supported by holidays library")
        return np.array([])
    
    if country_lower in holiday_mappings:
        important_days = holiday_mappings[country_lower]
    else:
        # Fallback - try to get all holidays if we don't have specific mapping
        important_days = set()
        for name in country_holidays.values():
            important_days.add(name.lower())
    
    days_off = [
        date.strftime("%Y-%m-%d")
        for date, name in country_holidays.items()
        if date.year == year and name.lower() in important_days
    ]
    
    return np.array(days_off)

In [ ]:
def is_european_holiday(date, country_code, holidays_array=None):
    """
    Check if a date is a public holiday in a European country.
    Returns 1 if it's a holiday, 0 otherwise.
    """
    date = pd.to_datetime(date) if not isinstance(date, pd.Timestamp) else date
    
    if holidays_array is None:
        holidays_array = get_european_public_holidays(country_code, date.year)
    
    return 1 if date.strftime("%Y-%m-%d") in holidays_array else 0

In [ ]:
# Test the functions with updated mappings
print("Testing European holiday functions...")

# Test Sweden's National Day (June 6th - equivalent to Norway's May 17th)
sweden_holidays_2024 = get_european_public_holidays('sweden', 2024)
print(f"Sweden 2024 holidays: {len(sweden_holidays_2024)} days")
print(f"Sweden National Day 2024-06-06: {is_european_holiday('2024-06-06', 'sweden', sweden_holidays_2024)}")

# Test Germany's Unity Day (October 3rd - their national day)
germany_holidays_2024 = get_european_public_holidays('germany', 2024)
print(f"Germany 2024 holidays: {len(germany_holidays_2024)} days")
print(f"German Unity Day 2024-10-03: {is_european_holiday('2024-10-03', 'germany', germany_holidays_2024)}")

# Test France's Bastille Day (July 14th - their national day)
france_holidays_2024 = get_european_public_holidays('france', 2024)
print(f"France 2024 holidays: {len(france_holidays_2024)} days")
print(f"Bastille Day 2024-07-14: {is_european_holiday('2024-07-14', 'france', france_holidays_2024)}")

# Test a regular day
print(f"Regular day 2024-03-15 in Sweden: {is_european_holiday('2024-03-15', 'sweden', sweden_holidays_2024)}")

print("\\nSample holidays from each country:")
print(f"Sweden: {sweden_holidays_2024[:5]}")
print(f"Germany: {germany_holidays_2024[:5]}")
print(f"France: {france_holidays_2024[:5]}")